# Layer Normalization

Normalization in deep learning is the process of transforming data or model outputs so that they have specific statistical properties, typically:

- Mean = 0
- Variance = 1

This is usually done using:

$$
x' = \frac{x - \mu}{\sigma}
$$



---

## Where Normalization is Applied

Normalization can be applied to:

- Input data
- Hidden layer activations
- Output of intermediate layers

---

# Benefits of Normalization

## 1. Improved Training Stability

Without normalization, values flowing through layers can:

- Grow too large → **exploding gradients**
- Become too small → **vanishing gradients**

### Problem

- Large activations → unstable updates, weights may "break"
- Small activations → gradients become tiny, learning slows or stops

### Solution

Normalization keeps values:

- centered
- scaled

This ensures gradients stay in a stable range for learning.

---

## 2. Faster Convergence

Normalization allows the use of **higher learning rates**.

Without it:

- Loss surface becomes elongated
- Optimizer oscillates in narrow valleys
- Training becomes slow

With normalization:

- Loss landscape becomes smoother and more circular
- Optimizer reaches minimum faster
- Fewer training epochs required

---

## 3. Mitigating Internal Covariate Shift

As training progresses:

- Early layers update their weights
- This changes the distribution of inputs to later layers

So later layers constantly learn from a **moving distribution**

### Normalization fixes this by:

Keeping activations stable in:

- mean
- variance

This makes learning more consistent across layers.

---

## 4. Regularization Effect

Especially in Batch Normalization:

- Statistics are computed over mini-batches
- This introduces small random noise

This noise:

- reduces overfitting
- prevents reliance on specific neurons
- improves generalization

---

# Types of Normalization

## 1. Batch Normalization

- Normalizes across the batch dimension
- Depends on batch statistics
- Works well in CNNs and feed-forward networks

---

## 2. Layer Normalization

- Normalizes across features (within a single sample)
- Independent of batch size
- Commonly used in Transformers

# Batch Normalization

![](./images/img19.png)

---

In **Batch Normalization**, we normalize the pre-activation values (often denoted as \(z\)) across a mini-batch.

For example, given:

- \(z_1, z_2, z_3, \dots\)

we normalize them so that they have:

- Mean ≈ 0  
- Variance ≈ 1  

---

# What is being normalized?

We normalize:

$$
z_1, z_2, z_3
$$

i.e., the raw outputs before activation functions.

---

# Step 1: Normalization

Each value is transformed as:

$$
\hat{z} = \frac{z - \mu}{\sigma}
$$

where:

- \( \mu \) = mean of the batch  
- \( \sigma \) = standard deviation of the batch  

---

# Step 2: Scaling and Shifting

After normalization, we apply learnable parameters:

- Scaling: \( \gamma \)
- Shifting: \( \beta \)

$$
y = \gamma \hat{z} + \beta
$$

---

# Why Scaling and Shifting?

Even though normalization forces values into a standard distribution, we still want the network to:

- recover original distributions if needed
- learn optimal transformations for the task

So:

- \( \gamma \) controls spread (variance)
- \( \beta \) controls shift (mean)

---

# Final Output Flow

1. Compute batch mean \( \mu \)
2. Compute batch variance \( \sigma^2 \)
3. Normalize:
   $$
   \hat{z} = \frac{z - \mu}{\sigma}
   $$
4. Scale and shift:
   $$
   y = \gamma \hat{z} + \beta
   $$

---

# Key Idea

Batch Normalization ensures that:

- activations stay stable across training
- gradients flow smoothly
- training becomes faster and more reliable

It is widely used in deep neural networks to improve performance and stability.

# Why Batch Normalization is NOT Used in Transformers

Batch Normalization does not work well with Transformers and sequential data.

The main reason is that it depends on **batch-level statistics**, which creates problems for variable-length sequences and self-attention.

---

# Self-Attention and Batching

In Transformers, we process multiple sentences together in a batch.

To do this, we:

- Stack multiple sentences together
- Pad shorter sentences with zero vectors so all sentences have equal length

Example:

```text
Sentence 1: Man killed lion
Sentence 2: Man killed a lion
```

After padding:

```text
Sentence 1: Man killed lion   [PAD]
Sentence 2: Man killed a lion
```

Padding tokens have embeddings of **0**.

---

# How Batch Normalization Works Here

Batch Normalization stacks all token embeddings from all sentences into a single matrix.

So instead of treating sentences independently, it mixes:

- all words
- all padding tokens
- all positions in the batch

![](./img20.png)
![](./img21.png)

---

# Why This Causes Problems

## 1. Padding Distortion

Because padding tokens are included in the batch statistics:

- many zeros affect mean and variance
- this distorts normalization values

So real word embeddings get incorrectly scaled.

---

## 2. Variable-Length Sequences

Different sentences have different lengths:

- some are short → more padding
- some are long → less padding

This creates inconsistent statistics across batches.

So the same word can get different normalized values depending on:

- sentence length
- batch composition

---

## 3. Breaks Sequential Independence

Self-Attention requires:

- each sentence to be treated independently
- no interference between sequences

But BatchNorm mixes statistics across the entire batch, which breaks this assumption.

---

## 4. Unstable Training Behavior

Because batch statistics change every iteration:

- outputs become noisy
- training becomes unstable
- convergence becomes harder

---

# Key Insight

Batch Normalization assumes:

> “All samples in a batch come from the same distribution.”

This assumption fails for NLP sequences because:

- sentences vary in length
- padding introduces artificial values
- attention requires per-token consistency

---

# What is Used Instead?

Transformers use:

## Layer Normalization

Because it:

- normalizes across features (not batch)
- works per sentence independently
- is stable for variable-length sequences
- works perfectly with self-attention

---

# Final Summary

Batch Normalization fails in Transformers because:

- it depends on batch-level statistics
- padding distorts normalization
- sentence lengths vary
- self-attention requires independent sequence processing

So instead, Transformers rely on **Layer Normalization**, which avoids these issues completely.

# Layer Normalization

In **Layer Normalization**, we normalize across the **features of a single token**, instead of normalizing across the batch.

---

# What does that mean?

In a Transformer, each word (token) is represented as a vector:

```text
x = [x1, x2, x3, ..., xd]
```

Instead of mixing values across different sentences (like BatchNorm), LayerNorm works **within this single vector**.

---

# How Layer Normalization Works

For each token embedding:

### Step 1: Compute mean and variance

$$
\mu = \frac{1}{d} \sum_{i=1}^{d} x_i
$$

$$
\sigma^2 = \frac{1}{d} \sum_{i=1}^{d} (x_i - \mu)^2
$$

---

### Step 2: Normalize

$$
\hat{x}_i = \frac{x_i - \mu}{\sigma}
$$

---

### Step 3: Scale and Shift

$$
y_i = \gamma \hat{x}_i + \beta
$$

where:

- \( \gamma \) = learnable scale parameter  
- \( \beta \) = learnable shift parameter  

---

# Key Idea

LayerNorm treats each token independently.

So:

- Each word is normalized on its own
- No dependency on batch size
- No interference from other sentences

---

# Why This Solves the “Zero Problem”

In Batch Normalization:

- padding tokens (0 values) affect batch statistics
- different sentence lengths distort mean/variance

But in Layer Normalization:

- we only look at one token at a time
- padding does NOT affect statistics of other words

So the “zero-padding problem” is avoided.

---

# Intuition

| Method | What is normalized? | Problem in NLP |
|--------|---------------------|----------------|
| BatchNorm | Across batch | Sensitive to padding & sentence length |
| LayerNorm | Across features (per token) | Stable for sequences |

---

# Why Transformers Use LayerNorm

Layer Normalization works well because:

- independent of batch size
- stable for variable-length sequences
- compatible with self-attention
- avoids padding-related distortion

---

# Final Summary

Layer Normalization:

- normalizes within each token vector
- does not depend on batch statistics
- removes issues caused by padding
- is the standard normalization used in Transformers